# Naive Bayes

In our previous lessons, we relied on Geometry (KNN) and Linear Algebra (SVM) to draw boundaries between data points. But what happens when our data isn't a continuous number like "CPU Load" or "Server RAM"? What happens when our data is human language—emails, tweets, or product reviews?

Drawing a geometric line through the English language is incredibly difficult. Instead, we must pivot to **Probability Theory**. 

In this lesson, we will explore **Naive Bayes**, a classification algorithm built on a 250-year-old mathematical theorem that remains one of the fastest and most effective ways to classify text in the enterprise.

Naive Bayes is a probabilistic classifier based on Bayes' Theorem. It calculates the probability that a data point belongs to a specific class given the evidence (the features). It is famously used for Spam Detection, Sentiment Analysis, and Document Categorization.

Let's set up our Python environment to build a foundational Spam Filter.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ Probabilistic NLP Environment Ready.")

✅ Probabilistic NLP Environment Ready.


# 1. The Mathematics of Bayes' Theorem

To understand Naive Bayes, we must first understand **Bayes' Theorem**. This formula allows us to update our beliefs based on new evidence.

$$P(y \mid X) = \frac{P(X \mid y) \cdot P(y)}{P(X)}$$

Let's break down these terms in the context of an email Spam Filter ($y = \text{Spam}$, $X = \text{The words in the email}$):
*   **$P(y \mid X)$ [The Posterior]:** The probability that the email is Spam *given* that we just saw the words in it. (This is the answer we want).
*   **$P(y)$ [The Prior]:** The baseline probability of an email being Spam before we even open it. (e.g., Historically, 20% of all emails are spam).
*   **$P(X \mid y)$ [The Likelihood]:** If we *know* an email is Spam, what is the probability of seeing these exact words?
*   **$P(X)$ [The Evidence]:** The total probability of seeing these exact words across *all* emails. (Since this is the same for both the Spam and Not-Spam calculations, we usually ignore it to save compute time).

Therefore, our algorithm simply calculates:
$$\text{Posterior} \propto \text{Prior} \times \text{Likelihood}$$

# 2. Why is it "Naive"?

Imagine an email has 100 words. Calculating the exact joint probability $P(\text{Word}_1, \text{Word}_2, \dots, \text{Word}_{100} \mid \text{Spam})$ is mathematically impossible because the order and combination of words are infinitely complex.

To solve this, the algorithm makes a **Naive Assumption**: It assumes that every single feature (word) is completely **conditionally independent** of every other feature. It pretends that the word "Credit" has absolutely no relationship to the word "Card". 

Because of this naive assumption, we can replace the complex joint probability with simple multiplication:
$$P(X \mid y) \approx P(x_1 \mid y) \times P(x_2 \mid y) \times \dots \times P(x_n \mid y)$$

*Does this assumption hold true in real life?* Absolutely not. Words are highly correlated. But remarkably, even though the core assumption is entirely false, Naive Bayes still performs exceptionally well in practice.

# 3. The Math Traps: Underflow and Zero-Frequency

Before we write the code, we must solve two fatal mathematical traps that occur when programming this algorithm.

### Trap 1: Mathematical Underflow (The Log Solution)
If an email has 200 words, we are multiplying 200 tiny probabilities together (e.g., $0.01 \times 0.05 \times 0.002 \dots$). The resulting number becomes so microscopic that a computer's CPU literally cannot store it, and it rounds the answer to `0.0`. This is called Underflow.

**The Solution:** We take the Natural Logarithm ($\log$) of the equation. A fundamental law of logarithms is that $\log(A \times B) = \log(A) + \log(B)$. Multiplication becomes Addition, saving our CPU!
$$\log P(y \mid X) \propto \log P(y) + \sum_{i=1}^{n} \log P(x_i \mid y)$$

### Trap 2: The Zero-Frequency Problem (Laplace Smoothing)
Suppose the word "Bitcoin" appears in a new email, but it *never* appeared in the "Legitimate" emails during training. The algorithm calculates $P(\text{"Bitcoin"} \mid \text{Legitimate}) = 0$. Because we are multiplying, that single zero destroys the entire equation, forcing the total probability to 0.

**The Solution:** **Laplace Smoothing ($\alpha$)**. We artificially add a count of $1$ (or $\alpha$) to every single word in the vocabulary so that a probability of absolute zero is mathematically impossible.

# 4. Implementing an NLP Text Classifier

Let's simulate a corporate dataset of emails, convert the text into a mathematical matrix (Word Counts), and train a `MultinomialNB` model.


In [2]:
# 1. Simulate Corporate Emails (Text Data)
emails = [
    "Win a free iPhone now, click here for money",       # Spam (1)
    "Urgent: Send money to my account to claim prize",   # Spam (1)
    "Free limited time offer, win cash",                 # Spam (1)
    "Hey team, when is the meeting tomorrow?",           # Ham  (0)
    "Please review the attached Q3 financial report",    # Ham  (0)
    "Lunch is in the breakroom, see you there",          # Ham  (0)
    "Meeting notes attached, great work on the project"  # Ham  (0)
]
labels = [1, 1, 1, 0, 0, 0, 0] # 1 = Spam, 0 = Ham

# 2. Feature Engineering: Convert Text to a Math Matrix
# CountVectorizer counts the frequency of each word, creating our Feature Matrix (X)
vectorizer = CountVectorizer(stop_words='english')
X_matrix = vectorizer.fit_transform(emails)
feature_names = vectorizer.get_feature_names_out()

# 3. Train the Naive Bayes Model
# alpha=1.0 applies Laplace Smoothing automatically!
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_matrix, labels)

# 4. Extract and Visualize the Math (Log Probabilities)
# We want to see which words the algorithm learned are "Spammy"
log_prob_spam = nb_model.feature_log_prob_[1]
log_prob_ham = nb_model.feature_log_prob_[0]

# Calculate the difference to find the most polarizing words
word_importance = log_prob_spam - log_prob_ham
sorted_indices = np.argsort(word_importance)

top_ham_words = [feature_names[i] for i in sorted_indices[:4]]
top_spam_words = [feature_names[i] for i in sorted_indices[-4:]]

print(f"Top indicators of Legitimate Email: {top_ham_words}")
print(f"Top indicators of Spam Email:       {top_spam_words[::-1]}")

# 5. Predict on a NEW, unseen email
new_email = ["Urgent meeting regarding the free project"]
new_email_matrix = vectorizer.transform(new_email)

prob = nb_model.predict_proba(new_email_matrix)
print(f"\n--- Prediction for: '{new_email[0]}' ---")
print(f"Probability of Legitimate: {prob[0][0] * 100:.1f}%")
print(f"Probability of Spam:       {prob[0][1] * 100:.1f}%")

Top indicators of Legitimate Email: ['attached', 'meeting', 'financial', 'breakroom']
Top indicators of Spam Email:       ['win', 'free', 'money', 'time']

--- Prediction for: 'Urgent meeting regarding the free project' ---
Probability of Legitimate: 57.1%
Probability of Spam:       42.9%


*(Insight: Notice how the algorithm classified the new email. It contains "Spam" words ('Urgent', 'Free') and "Ham" words ('Meeting', 'Project'). The algorithm calculated the Prior, pulled the exact Likelihood probabilities for those 4 specific words from its memory, added the Logarithms together, and confidently output the final Posterior probability!)*

## Real-World Use Case or Analogy:
Think of Naive Bayes like **A Detective Solving a Crime**:

*   **The Prior ($P(y)$)**: The detective knows that historically, 80% of crimes in this neighborhood are committed by the local gang, and 20% by outsiders. Before looking at *any* clues, the gang is the primary suspect.
*   **The Likelihood ($P(X \mid y)$)**: The detective finds a cigar ($x_1$) and a muddy boot ($x_2$). They check their historical records: "If the gang committed the crime, how often do they leave cigars?" (Very often). "If outsiders did it, how often do they leave cigars?" (Rarely).
*   **The Naive Assumption**: The detective treats the cigar and the boot as completely independent clues. They do not pause to consider that maybe muddy boots specifically *cause* dropped cigars. They just multiply the isolated probabilities.
*   **The Posterior ($P(y \mid X)$)**: By combining the baseline historical rate (Prior) with the specific evidence at the scene (Likelihood), the detective calculates a final 95% certainty that the gang committed the crime.

---